# Reproduce the iMSE benchmark in Colab

Choose a dataset and optionally one backbone. Run the cells in order for download, feature extraction, and evaluation. The default `STAGE="all"` runs the complete benchmark, not a sample run. Use `STAGE="evaluate"` to reuse completed features.

Data come from the original public sources, including OpenOOD's public Drive links. Your personal Drive and the authors' saved experiments are not required. ImageNet-1K needs accepted [access terms](https://huggingface.co/datasets/ILSVRC/imagenet-1k) and an `HF_TOKEN`. Full ImageNet-1K needs substantial disk and compute. See the [README](README.md) for sources and protocol.

Every command displays output and saves a timestamped log under `WORK/logs`. If a step fails, its actual error appears before the final exception. Completed data and feature caches are retained. After changing benchmark code, use a new results directory as explained in the README.

Select a GPU in **Runtime → Change runtime type**. Add a read token named `HF_TOKEN` to **Colab Secrets** for gated downloads. `/content` storage is temporary; copy outputs before deleting the runtime, or enable the optional Drive storage setting below.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DATASET = "CIFAR10"  # CIFAR10, CIFAR100, ImageNet1K
BACKBONE = ""  # Empty: all paper backbones; e.g. ResNet18 for CIFAR10
STAGE = "all"  # all, download, extract, evaluate
DEVICE = "cuda"  # cpu is also supported except for Vim-Base

REPO = Path("/content/iMSE")
WORK = Path("/content/iMSE-work")
UPDATE_REPO = True
SAVE_TO_DRIVE = False  # Optional output storage, never an input-data dependency

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/iMSE-work")


In [ ]:
def run(command, label):
    from datetime import datetime, timezone

    log_path = WORK / "logs" / f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}-{label}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env.setdefault("TQDM_MININTERVAL", "10")
    print(f"{label} | log: {log_path}", flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=REPO if REPO.exists() else None,
                                   env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace")
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
        finally:
            process.stdout.close()
    if code:
        raise RuntimeError(f"{label} exited with code {code}. See the output above or {log_path}.")


## Install and check the environment

In [ ]:
from google.colab import userdata

if not os.environ.get("HF_TOKEN"):
    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
    if token:
        os.environ["HF_TOKEN"] = token
    del token

if not REPO.exists():
    run(["git", "clone", "https://github.com/erantascesme/iMSE.git", str(REPO)], "clone")
elif UPDATE_REPO:
    status = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True)
    if status.strip():
        raise RuntimeError("The checkout has local changes. Preserve them first, or set UPDATE_REPO=False.")
    run(["git", "pull", "--ff-only", "origin", "main"], "update")
if not (REPO / "imse_benchmark").is_dir():
    raise RuntimeError(f"Not an iMSE checkout: {REPO}")
run(["git", "rev-parse", "HEAD"], "revision")

needs_vim = (DATASET == "ImageNet1K" and BACKBONE in {"", "Vim-Base"}
             and STAGE in {"all", "extract"})
package = f"{REPO}[vim]" if needs_vim else str(REPO)
run([sys.executable, "-m", "pip", "install", "-e", package], "install")

# Inspect the manifest without importing packages that pip may have just updated.
import json
checkpoints = json.loads((REPO / "imse_benchmark" / "assets" / "checkpoints.json").read_text())
models = [key.split("/", 1)[1] for key in checkpoints if key.startswith(DATASET + "/")]
if STAGE not in {"all", "download", "extract", "evaluate"} or not models:
    raise ValueError("Choose a valid DATASET and STAGE.")
if BACKBONE and BACKBONE not in models:
    raise ValueError(f"Choose a backbone from {models}, or leave BACKBONE empty.")
print("Backbones:", models if not BACKBONE else [BACKBONE])
print("Work directory:", WORK)
run([sys.executable, "-m", "pip", "show", "torch", "torchvision", "timm"], "versions")
if STAGE != "download":
    check = "import torch, torchvision, timm, faiss; print('PyTorch:', torch.__version__); "
    if DEVICE.startswith("cuda"):
        check += "assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0))"
    run([sys.executable, "-c", check], "environment")


### Optional: use an already downloaded original archive

Leave `MANUAL_ARCHIVES` empty for normal downloads. If an OpenOOD public link exceeds its download quota, provide an **exact original ZIP** here instead of waiting for the quota to reset. In Colab, upload it through the Files sidebar to `/content`, then enter its path below. This does not require mounting Google Drive. Rerun this cell and **Download datasets**; archive hashes and benchmark splits remain unchanged. Do not unzip or repack the file.

In [ ]:
MANUAL_ARCHIVES = []  # e.g. ["/content/cifar10.zip"] in Colab

import hashlib
import json
import shutil

specs = json.loads((REPO / "imse_benchmark" / "assets" / "datasets.json").read_text())
archives = specs[DATASET].get("archives", {})
for filename in MANUAL_ARCHIVES:
    source = Path(filename).expanduser().resolve()
    spec = archives.get(source.stem)
    if source.suffix != ".zip" or spec is None:
        raise ValueError(f"Expected an original CIFAR benchmark archive: {list(archives)}")
    with source.open("rb") as handle:
        digest = hashlib.file_digest(handle, "sha256").hexdigest()
    if digest != spec["sha256"]:
        raise ValueError(f"{source.name} differs from the original benchmark archive. Do not use it.")
    destination = WORK / "data" / "cifar" / source.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source != destination.resolve():
        temporary = destination.with_suffix(".importing")
        shutil.copyfile(source, temporary)
        with temporary.open("rb") as handle:
            if hashlib.file_digest(handle, "sha256").hexdigest() != digest:
                raise RuntimeError("Archive copy failed verification")
        os.replace(temporary, destination)
    print("Verified original archive:", destination)


## Download datasets

Downloads the original public benchmark archives. No personal Google Drive is needed.

In [ ]:
if STAGE in {"all", "download"}:
    args = [sys.executable, "-u", "-m", "imse_benchmark", "download", "--dataset", DATASET,
            "--work-dir", str(WORK), "--device", DEVICE]
    if BACKBONE:
        args += ["--backbone", BACKBONE]
    run(args, DATASET + "-download")

## Extract features

Loads the released checkpoints and retains the original preprocessing, split order, and extraction batch sizes. Completed splits are reused.

In [ ]:
if STAGE in {"all", "extract"}:
    args = [sys.executable, "-u", "-m", "imse_benchmark", "extract", "--dataset", DATASET,
            "--work-dir", str(WORK), "--device", DEVICE]
    if BACKBONE:
        args += ["--backbone", BACKBONE]
    run(args, DATASET + "-extract")

## Evaluate detectors

Fits on ID training data and evaluates every paper detector. Prints All, Near, and Far tables, with metrics saved under `WORK/results`.

In [ ]:
if STAGE in {"all", "evaluate"}:
    args = [sys.executable, "-u", "-m", "imse_benchmark", "evaluate", "--dataset", DATASET,
            "--work-dir", str(WORK), "--device", DEVICE]
    if BACKBONE:
        args += ["--backbone", BACKBONE]
    run(args, DATASET + "-evaluate")